# rfsoc4x2 — first run

Describe the setup, connect, and play one readout. Units are SI: Hz, seconds,
degrees, gain in -1 .. 1.


In [ ]:
from pathlib import Path
from qcodes import Station
import numpy as np
import matplotlib.pyplot as plt

# all from the rfsoc4x2 package
from rfsoc4x2 import RFSoC, Config, ResonatorSpec, ConstantPulseSpec, RunConfig
from rfsoc4x2.sequences import readout

from qcodes.dataset import (load_by_run_spec, load_by_id)
from qcodes.interactive_widget import experiments_widget

In [ ]:
IP_ADDRESS = "192.168.2.99"
path2db = Path("database/test.db")

## Setup

An element owns a DAC, a carrier frequency and (for a resonator) an ADC.
A pulse takes its DAC and carrier from its element, so it only says how far
it sits from that carrier.


In [ ]:
resonator = ResonatorSpec(name='resonator', dac=1, adc=2, frequency=5e9, nqz=2, readout_length=1e-6, time_of_flight=0e-9)
readout_pulse = ConstantPulseSpec(name='readout', element='resonator', detuning=0, phase=0, gain=1, length=0.5e-6)

conf = Config(elements = [resonator], pulses = [readout_pulse])

## Connect

Building the RFSoC configures the channels and creates every pulse.
The board's pyro server must register as `myqick`; pass `ns_port` if it is not 8888.


In [ ]:
station = Station()

rfsoc = RFSoC(name="rfsoc", ip=IP_ADDRESS, station=station, db_path=path2db, config=conf)

# tProc type and revision, sample rates, generator types, marker pins
print(rfsoc.qi.soccfg)


## Run

`RunConfig` holds everything that belongs to the run rather than to the setup.
Start in `decimated` mode to see the waveform and read the real time of flight.


In [ ]:
measurement_name = "test_measurement"
experiment_name = "test_experiment"
sample_name = "test_sample"

run_config = RunConfig(
    measurement_name=measurement_name,
    experiment_name=experiment_name,
    sample_name=sample_name,
    acquisition_mode="decimated",
    n_shots=20,
    soft_avgs=100,
)

In [ ]:
run_id = rfsoc.run(readout(rfsoc.element("resonator")), config=run_config)

In [ ]:
experiments_widget()

## Recover the data

`run()` returns the qcodes run id. In `accumulated` mode the result is a single
complex point; in `decimated` mode it is a trace, which is what you want here to
read the time of flight off.


In [ ]:
dataset = load_by_id(run_id)
data = dataset.get_parameter_data()

name = list(data)[0]                    # 'iq', or 'iq_ch2' with several ADCs
block = data[name]
iq = np.asarray(block[name]).ravel()


print(dataset.name, '| run', dataset.run_id, '| captured:', list(block))
print('points:', iq.size)


## Show it


In [ ]:
if 'time' in block:
    t = np.asarray(block['time']).ravel()

    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
    ax1.plot(t * 1e9, np.abs(iq))
    ax1.set_ylabel('|IQ| (ADC units)')
    ax1.axvline(resonator.time_of_flight * 1e9, color='k', ls='--', lw=1,
                label='configured time of flight')
    ax1.legend()

    ax2.plot(t * 1e9, np.unwrap(np.angle(iq)) * 180 / np.pi)
    ax2.set_ylabel('phase (deg)')
    ax2.set_xlabel('time (ns)')
    fig.suptitle(f'{dataset.name} (run {dataset.run_id})')
    plt.tight_layout()

    # where the pulse actually arrives -> put this into time_of_flight
    above = np.abs(iq) > 0.5 * np.abs(iq).max()
    print(f'pulse arrives at {t[np.argmax(above)] * 1e9:.0f} ns')
else:
    # accumulated with no sweeps: one complex number
    point = iq[0]
    print(f'I = {point.real:.5f}   Q = {point.imag:.5f}')
    print(f'|IQ| = {np.abs(point):.5f}   phase = {np.angle(point, deg=True):.2f} deg')


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12,8), sharex=True)

time = block["time"].ravel()

iq_mod = np.abs(iq)
phase = np.unwrap(np.rad2deg(np.angle(iq)))

epsilon = 20
margin = 2e-9

index = np.where(iq_mod > epsilon)[0]

time_of_flight = time[index[0]] + margin


print(index)



ax[0].plot(time, iq_mod)
ax[0].axvline(time_of_flight, linestyle="--", color="black")
ax[1].plot(time, phase)

In [ ]:
def detect_arrival(t, iq, pulse_length, smooth=7, min_snr=10):
    """Pulse arrival time (s) in a decimated trace, measured from the ADC trigger."""
    t = np.ravel(t)
    amp = np.convolve(np.abs(np.ravel(iq)), np.ones(smooth) / smooth, mode="same")
    # split the samples into "no signal" and "signal", threshold halfway between them
    thr = 0.5 * (amp.min() + amp.max())
    for _ in range(100):
        low, high = amp[amp <= thr].mean(), amp[amp > thr].mean()
        thr, old = 0.5 * (low + high), thr
        if abs(thr - old) < 1e-9 * high:
            break
    snr = (high - low) / amp[amp <= thr].std()
    above = amp > thr
    if snr < min_snr:
        raise RuntimeError(f"no clear pulse (S/N {snr:.1f}): average more")
    if above[:smooth].any():
        raise RuntimeError("pulse already there when the ADC starts: trigger at the pulse start")
    if above[-smooth:].any():
        raise RuntimeError("pulse still on when the window ends: use a longer window")
    i = np.argmax(above)                              # first sample above the threshold
    j = len(above) - 1 - np.argmax(above[::-1])       # last sample above the threshold
    arrival = np.interp(thr, amp[i - 1:i + 1], t[i - 1:i + 1])  # between the two samples
    width = t[j] - t[i]
    if abs(width - pulse_length) > 0.2 * pulse_length:
        raise RuntimeError(f"pulse looks {width*1e9:.0f} ns long, expected {pulse_length*1e9:.0f} ns")
    return arrival

In [ ]:
arrival = detect_arrival(time, iq, readout_pulse.length)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12,8), sharex=True)

time = block["time"].ravel()

iq_mod = np.abs(iq)
phase = np.unwrap(np.rad2deg(np.angle(iq)))


ax[0].plot(time, iq_mod)
ax[0].axvline(arrival, linestyle="--", color="black")
ax[1].plot(time, phase)

## Autocalibration

In [ ]:
from rfsoc4x2.programs.tof_calibration import tof_calibration

tof, run_id = tof_calibration(
    rfsoc=rfsoc,
    resonator=rfsoc.element("resonator"),
    pulse_length=readout_pulse.length,
    window=1e-6,
    margin=5e-9,
    smooth=7,
    min_snr=10,
    soft_avg=10,
    verbose=True,
    plot=True
)